In [55]:
from pathlib import Path
import zipfile
import pandas as pd
import numpy as np
import re

### Locate the original data file

I first check that the Companies House ZIP file is available in the raw data folder.

In [62]:
original_data_folder = Path("../data/raw")
for file in original_data_folder.iterdir():
    print(file.name)

BasicCompanyDataAsOneFile-2026-08-01.zip
defra_spending_january_2026.csv


### Check the contents of the downloaded archive

Before reading the dataset, I check which file is stored inside the ZIP archive.

In [63]:
companies_zip_file = original_data_folder / "BasicCompanyDataAsOneFile-2026-08-01.zip"

with zipfile.ZipFile(companies_zip_file, "r") as zip_file:
    files_inside_zip = zip_file.namelist()

files_inside_zip

['BasicCompanyDataAsOneFile-2026-08-01.csv']

### Preview the company data

I first load only the first 5 rows of the Companies House dataset.
The goal is to understand the structure and available columns before processing the full file.

In [64]:
companies_preview = pd.read_csv(
    companies_zip_file,
    compression="zip",
    nrows=5,
    dtype={"CompanyNumber": str}
)

companies_preview.columns = companies_preview.columns.str.strip()

companies_preview
companies_preview

,CompanyName,CompanyNumber,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,PreviousName_7.CONDATE,PreviousName_7.CompanyName,PreviousName_8.CONDATE,PreviousName_8.CompanyName,PreviousName_9.CONDATE,PreviousName_9.CompanyName,PreviousName_10.CONDATE,PreviousName_10.CompanyName,ConfStmtNextDueDate,ConfStmtLastMadeUpDate
0,! LTD,8209948,NaN,NaN,9 PRINCES SQUARE,NaN,HARROGATE,NaN,ENGLAND,HG1 1ND,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25/09/2026,11/09/2025
1,!ABRIDGE TAX LTD,16092999,NaN,NaN,82 GREAT NORTH ROAD,GREAT NORTH BUSINESS CENTRE,HATFIELD,NaN,UNITED KINGDOM,AL9 5BL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,04/12/2026,20/11/2025
2,!BIG IMPACT GRAPHICS LIMITED,11743365,NaN,NaN,124 CITY ROAD,NaN,LONDON,NaN,NaN,EC1V 2NX,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29/12/2026,15/12/2025
3,!FE NETWORK LIMITED,16873705,NaN,NaN,C/O AACSL ACCOUNTANT LIMITED,1ST FLOOR NORTH WESTGATE HOUSE,"HARLOW, ESSEX",NaN,UNITED KINGDOM,CM20 1YS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,08/12/2026,NaN
4,!NFLECTION ADVISORY LIMITED,15073164,NaN,NaN,74 SANTERS LANE,NaN,POTTERS BAR,HERTFORDSHIRE,ENGLAND,EN6 2DA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28/08/2026,14/08/2025


### Inspect the available columns

I check the column names to understand what company information is available and which fields may be useful later for sampling and company matching.

In [65]:
companies_preview.columns.tolist()

['CompanyName',
 'CompanyNumber',
 'RegAddress.CareOf',
 'RegAddress.POBox',
 'RegAddress.AddressLine1',
 'RegAddress.AddressLine2',
 'RegAddress.PostTown',
 'RegAddress.County',
 'RegAddress.Country',
 'RegAddress.PostCode',
 'CompanyCategory',
 'CompanyStatus',
 'CountryOfOrigin',
 'DissolutionDate',
 'IncorporationDate',
 'Accounts.AccountRefDay',
 'Accounts.AccountRefMonth',
 'Accounts.NextDueDate',
 'Accounts.LastMadeUpDate',
 'Accounts.AccountCategory',
 'Returns.NextDueDate',
 'Returns.LastMadeUpDate',
 'Mortgages.NumMortCharges',
 'Mortgages.NumMortOutstanding',
 'Mortgages.NumMortPartSatisfied',
 'Mortgages.NumMortSatisfied',
 'SICCode.SicText_1',
 'SICCode.SicText_2',
 'SICCode.SicText_3',
 'SICCode.SicText_4',
 'LimitedPartnerships.NumGenPartners',
 'LimitedPartnerships.NumLimPartners',
 'URI',
 'PreviousName_1.CONDATE',
 'PreviousName_1.CompanyName',
 'PreviousName_2.CONDATE',
 'PreviousName_2.CompanyName',
 'PreviousName_3.CONDATE',
 'PreviousName_3.CompanyName',
 'Previou

### Clean the column names

Some column names contain extra spaces at the beginning or end. I remove these spaces so the fields can be referenced consistently during the analysis.

In [66]:
companies_preview.columns = companies_preview.columns.str.strip()

companies_preview.columns.tolist()

['CompanyName',
 'CompanyNumber',
 'RegAddress.CareOf',
 'RegAddress.POBox',
 'RegAddress.AddressLine1',
 'RegAddress.AddressLine2',
 'RegAddress.PostTown',
 'RegAddress.County',
 'RegAddress.Country',
 'RegAddress.PostCode',
 'CompanyCategory',
 'CompanyStatus',
 'CountryOfOrigin',
 'DissolutionDate',
 'IncorporationDate',
 'Accounts.AccountRefDay',
 'Accounts.AccountRefMonth',
 'Accounts.NextDueDate',
 'Accounts.LastMadeUpDate',
 'Accounts.AccountCategory',
 'Returns.NextDueDate',
 'Returns.LastMadeUpDate',
 'Mortgages.NumMortCharges',
 'Mortgages.NumMortOutstanding',
 'Mortgages.NumMortPartSatisfied',
 'Mortgages.NumMortSatisfied',
 'SICCode.SicText_1',
 'SICCode.SicText_2',
 'SICCode.SicText_3',
 'SICCode.SicText_4',
 'LimitedPartnerships.NumGenPartners',
 'LimitedPartnerships.NumLimPartners',
 'URI',
 'PreviousName_1.CONDATE',
 'PreviousName_1.CompanyName',
 'PreviousName_2.CONDATE',
 'PreviousName_2.CompanyName',
 'PreviousName_3.CONDATE',
 'PreviousName_3.CompanyName',
 'Previou

### Select the most useful company information

The dataset contains many columns, but only some of them are relevant for this project. I keep the fields that can help identify a company, describe it, and later compare it with VAT information found from other sources..

In [67]:
useful_columns = [
    "CompanyName",
    "CompanyNumber",
    "CompanyStatus",
    "CompanyCategory",
    "IncorporationDate",
    "RegAddress.AddressLine1",
    "RegAddress.AddressLine2",
    "RegAddress.PostTown",
    "RegAddress.County",
    "RegAddress.Country",
    "RegAddress.PostCode",
    "SICCode.SicText_1"
]

companies_preview[useful_columns]

,CompanyName,CompanyNumber,CompanyStatus,CompanyCategory,IncorporationDate,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,SICCode.SicText_1
0,! LTD,8209948,Active,Private Limited Company,11/09/2012,9 PRINCES SQUARE,NaN,HARROGATE,NaN,ENGLAND,HG1 1ND,99999 - Dormant Company
1,!ABRIDGE TAX LTD,16092999,Active,Private Limited Company,21/11/2024,82 GREAT NORTH ROAD,GREAT NORTH BUSINESS CENTRE,HATFIELD,NaN,UNITED KINGDOM,AL9 5BL,62020 - Information technology consultancy act...
2,!BIG IMPACT GRAPHICS LIMITED,11743365,Active,Private Limited Company,28/12/2018,124 CITY ROAD,NaN,LONDON,NaN,NaN,EC1V 2NX,59112 - Video production activities
3,!FE NETWORK LIMITED,16873705,Active,Private Limited Company,25/11/2025,C/O AACSL ACCOUNTANT LIMITED,1ST FLOOR NORTH WESTGATE HOUSE,"HARLOW, ESSEX",NaN,UNITED KINGDOM,CM20 1YS,86210 - General medical practice activities
4,!NFLECTION ADVISORY LIMITED,15073164,Active,Private Limited Company,15/08/2023,74 SANTERS LANE,NaN,POTTERS BAR,HERTFORDSHIRE,ENGLAND,EN6 2DA,70229 - Management consultancy activities othe...


### Check the data types

I inspect how the fields were read by Python. Company identifiers should be treated as text rather than numbers, because they are identifiers and may contain letters or leading zeros.

In [68]:
companies_preview[useful_columns].dtypes

CompanyName                  str
CompanyNumber              int64
CompanyStatus                str
CompanyCategory              str
IncorporationDate            str
RegAddress.AddressLine1      str
RegAddress.AddressLine2      str
RegAddress.PostTown          str
RegAddress.County            str
RegAddress.Country           str
RegAddress.PostCode          str
SICCode.SicText_1            str
dtype: object

### Treat the company number as an identifier

The company number was initially read as an integer. Since it is an identifier rather than a numeric value, I read it as text to preserve its original format.

In [69]:
companies_preview = pd.read_csv(
    companies_zip_file,
    compression="zip",
    nrows=5,
    skipinitialspace=True,
    dtype={"CompanyNumber": str}
)

companies_preview

,CompanyName,CompanyNumber,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,PreviousName_7.CONDATE,PreviousName_7.CompanyName,PreviousName_8.CONDATE,PreviousName_8.CompanyName,PreviousName_9.CONDATE,PreviousName_9.CompanyName,PreviousName_10.CONDATE,PreviousName_10.CompanyName,ConfStmtNextDueDate,ConfStmtLastMadeUpDate
0,! LTD,08209948,NaN,NaN,9 PRINCES SQUARE,NaN,HARROGATE,NaN,ENGLAND,HG1 1ND,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25/09/2026,11/09/2025
1,!ABRIDGE TAX LTD,16092999,NaN,NaN,82 GREAT NORTH ROAD,GREAT NORTH BUSINESS CENTRE,HATFIELD,NaN,UNITED KINGDOM,AL9 5BL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,04/12/2026,20/11/2025
2,!BIG IMPACT GRAPHICS LIMITED,11743365,NaN,NaN,124 CITY ROAD,NaN,LONDON,NaN,NaN,EC1V 2NX,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29/12/2026,15/12/2025
3,!FE NETWORK LIMITED,16873705,NaN,NaN,C/O AACSL ACCOUNTANT LIMITED,1ST FLOOR NORTH WESTGATE HOUSE,"HARLOW, ESSEX",NaN,UNITED KINGDOM,CM20 1YS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,08/12/2026,NaN
4,!NFLECTION ADVISORY LIMITED,15073164,NaN,NaN,74 SANTERS LANE,NaN,POTTERS BAR,HERTFORDSHIRE,ENGLAND,EN6 2DA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28/08/2026,14/08/2025


### Understand the company population

Before choosing a sample, I want to understand the companies available in the dataset. I count the total number of companies and check how they are distributed by company status.

Because the dataset is large, I read it in smaller parts instead of loading the entire file into memory.

In [70]:
total_companies = 0
company_status_counts = {}

for company_data_part in pd.read_csv(
    companies_zip_file,
    compression="zip",
    usecols=["CompanyStatus"],
    skipinitialspace=True,
    chunksize=100_000
):
    total_companies += len(company_data_part)

    status_counts = company_data_part["CompanyStatus"].value_counts()

    for status, count in status_counts.items():
        company_status_counts[status] = (
            company_status_counts.get(status, 0) + count
        )

In [71]:
print("Total companies:", total_companies)

company_status_summary = (
    pd.Series(company_status_counts)
    .sort_values(ascending=False)
)

company_status_summary

Total companies: 5695465


Active                                              5190464
Active - Proposal to Strike off                      388951
Liquidation                                          108797
In Administration                                      3739
Live but Receiver Manager on at least one charge       2119
Voluntary Arrangement                                   576
In Administration/Administrative Receiver               344
RECEIVERSHIP                                            182
ADMINISTRATION ORDER                                    121
ADMINISTRATIVE RECEIVER                                 105
In Administration/Receiver Manager                       54
RECEIVER MANAGER / ADMINISTRATIVE RECEIVER               10
VOLUNTARY ARRANGEMENT / RECEIVER MANAGER                  2
VOLUNTARY ARRANGEMENT / ADMINISTRATIVE RECEIVER           1
dtype: int64


Most companies in the dataset are active. Since the use case concerns suppliers that are currently doing business, I use only companies with the status `Active` for the main sample.

I exclude companies in liquidation, administration, or proposed for strike-off because they are less representative of the target supplier population.

In [72]:
active_companies = company_status_counts["Active"]

active_share = active_companies / total_companies * 100

print("Active companies:", active_companies)
print(f"Share of active companies: {active_share:.1f}%")

Active companies: 5190464
Share of active companies: 91.1%


### Select a random sample of active companies

For the proof of concept, I select 100 companies at random from all active Companies House records.

The sample is selected before searching for VAT numbers, so companies are not chosen based on whether their VAT information is easy to find.

I use a fixed random seed so that the same sample can be reproduced.

This sample represents active Companies House companies, not necessarily the exact supplier mix of a mid-sized manufacturer. Without access to the customer's supplier population, I avoid making assumptions about its industry distribution.

In [73]:
sample_size = 100
random_seed = 42

random_generator = np.random.default_rng(random_seed)

selected_company_positions = np.sort(
    random_generator.choice(
        active_companies,
        size=sample_size,
        replace=False
    )
)

selected_company_positions[:10]

array([227359, 331237, 352532, 396254, 446089, 463245, 478221, 488815,
       504650, 664958])

### Extract the selected companies

I read the Companies House data in smaller parts and keep only active companies whose positions were selected for the sample.

I retain the fields that will be useful for company identification, VAT discovery, and later validation.

In [74]:
columns_for_sample = [
    "CompanyName",
    "CompanyNumber",
    "CompanyStatus",
    "CompanyCategory",
    "IncorporationDate",
    "RegAddress.AddressLine1",
    "RegAddress.AddressLine2",
    "RegAddress.PostTown",
    "RegAddress.County",
    "RegAddress.Country",
    "RegAddress.PostCode",
    "SICCode.SicText_1"
]

selected_companies = []
active_position = 0

for company_data_part in pd.read_csv(
    companies_zip_file,
    compression="zip",
    usecols=columns_for_sample,
    skipinitialspace=True,
    dtype={"CompanyNumber": str},
    chunksize=100_000
):

    active_company_data = company_data_part[
        company_data_part["CompanyStatus"] == "Active"
    ].copy()

    number_of_active_companies = len(active_company_data)

    positions_in_this_part = selected_company_positions[
        (selected_company_positions >= active_position)
        & (
            selected_company_positions
            < active_position + number_of_active_companies
        )
    ]

    if len(positions_in_this_part) > 0:

        positions_inside_part = (
            positions_in_this_part - active_position
        )

        selected_companies.append(
            active_company_data.iloc[positions_inside_part]
        )

    active_position += number_of_active_companies

In [75]:
company_sample = pd.concat(
    selected_companies,
    ignore_index=True
)

print("Companies selected:", len(company_sample))

company_sample.head()

Companies selected: 100


,CompanyName,CompanyNumber,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,CompanyCategory,CompanyStatus,IncorporationDate,SICCode.SicText_1
0,AJ PARTITIONS AND CEILINGS LTD,14816463,PARKINS ACCOUNTANTS,"MOOR PARK HOUSE, BAWTRY ROAD",ROTHERHAM,SOUTH YORKSHIRE,ENGLAND,S66 2BL,Private Limited Company,Active,20/04/2023,43310 - Plastering
1,ANDREWS RESIDENTIAL LIMITED,12574830,"OFFICE 5, MARLOWE HOUSE WATLING STREET",HOCKLIFFE,LEIGHTON BUZZARD,NaN,ENGLAND,LU7 9LS,Private Limited Company,Active,28/04/2020,68209 - Other letting and operating of own or ...
2,AOB SUBSEA LTD,SC682059,4 WEST CRAIBSTONE STREET,(BON-ACCORD SQUARE),ABERDEEN,ABERDEENSHIRE,UNITED KINGDOM,AB11 6YL,Private Limited Company,Active,26/11/2020,71129 - Other engineering activities
3,ARLINGTON GROUP ASSET MANAGEMENT LIMITED,02359077,15 WHITEHALL,NaN,LONDON,NaN,NaN,SW1A 2DD,Private Limited Company,Active,10/03/1989,64999 - Financial intermediation not elsewhere...
4,ASTRAZENECA UK LIMITED,03674842,1 FRANCIS CRICK AVENUE,CAMBRIDGE BIOMEDICAL CAMPUS,CAMBRIDGE,NaN,UNITED KINGDOM,CB2 0AA,Private Limited Company,Active,26/11/1998,70100 - Activities of head offices


### Check the selected sample

I verify that the sample contains 100 unique companies and review a few records before using it for VAT discovery.

In [76]:
print("Number of companies:", len(company_sample))
print(
    "Unique company numbers:",
    company_sample["CompanyNumber"].nunique()
)

company_sample[
    [
        "CompanyName",
        "CompanyNumber",
        "IncorporationDate",
        "RegAddress.PostTown",
        "RegAddress.Country",
        "SICCode.SicText_1"
    ]
].head(10)

Number of companies: 100
Unique company numbers: 100


,CompanyName,CompanyNumber,IncorporationDate,RegAddress.PostTown,RegAddress.Country,SICCode.SicText_1
0,AJ PARTITIONS AND CEILINGS LTD,14816463,20/04/2023,ROTHERHAM,ENGLAND,43310 - Plastering
1,ANDREWS RESIDENTIAL LIMITED,12574830,28/04/2020,LEIGHTON BUZZARD,ENGLAND,68209 - Other letting and operating of own or ...
2,AOB SUBSEA LTD,SC682059,26/11/2020,ABERDEEN,UNITED KINGDOM,71129 - Other engineering activities
3,ARLINGTON GROUP ASSET MANAGEMENT LIMITED,02359077,10/03/1989,LONDON,NaN,64999 - Financial intermediation not elsewhere...
4,ASTRAZENECA UK LIMITED,03674842,26/11/1998,CAMBRIDGE,UNITED KINGDOM,70100 - Activities of head offices
5,AUDIO NUTRITION LTD,17027036,11/02/2026,BLACKPOOL,ENGLAND,59200 - Sound recording and music publishing a...
6,AV GLOBAL HOLDINGS LTD,17196916,04/05/2026,PETERBOROUGH,ENGLAND,49410 - Freight transport by road
7,AVTAR INVESTMENTS LTD,16376979,09/04/2025,WOLVERHAMPTON,ENGLAND,68209 - Other letting and operating of own or ...
8,B & D ELIAS PROPERTIES LTD,12885766,17/09/2020,SWANSEA,WALES,68209 - Other letting and operating of own or ...
9,BLAIRGOWRIE EVANGELICAL CHURCH SCIO,CS007183,31/10/2024,NaN,NaN,None Supplied


### Save the selected sample

I save the 100 selected companies before starting VAT discovery. This keeps the sample fixed and prevents later choices from being influenced by how easy or difficult a company's VAT number is to find.

In [77]:
processed_data_folder = Path("../data/processed")

sample_file = processed_data_folder / "company_sample_100.csv"

company_sample.to_csv(
    sample_file,
    index=False
)

print("Sample saved:", sample_file)

Sample saved: ..\data\processed\company_sample_100.csv


### Understand the selected sample

Before starting VAT discovery, I review the sample to understand what kinds of companies it contains and whether important identification information is missing.

In [78]:
company_sample["RegAddress.Country"].value_counts(dropna=False)

RegAddress.Country
ENGLAND             58
UNITED KINGDOM      19
NaN                 16
SCOTLAND             3
WALES                2
NORTHERN IRELAND     2
Name: count, dtype: int64

### Observation

The registered country field is not fully consistent across the sample. Some companies are recorded as `England`, others as `United Kingdom`, while 16 companies have no country value.

I keep these records in the sample because a missing country field does not by itself mean that the company is unsuitable for the analysis. Other address fields, such as postcode and town, may still help identify the company later.

### Review company types

I check the legal categories represented in the sample to see whether the random selection contains different types of active companies.

In [79]:
company_sample["CompanyCategory"].value_counts()

CompanyCategory
Private Limited Company                                                  95
Scottish Charitable Incorporated Organisation                             2
Community Interest Company                                                1
Public Limited Company                                                    1
PRI/LTD BY GUAR/NSC (Private, limited by guarantee, no share capital)     1
Name: count, dtype: int64

### Review business activities

I inspect the SIC descriptions to understand the range of business activities represented in the random sample.

In [80]:
company_sample["SICCode.SicText_1"].value_counts().head(15)

SICCode.SicText_1
68209 - Other letting and operating of own or leased real estate             9
70229 - Management consultancy activities other than financial management    5
82990 - Other business support service activities n.e.c.                     4
43390 - Other building completion and finishing                              4
68100 - Buying and selling of own real estate                                4
41100 - Development of building projects                                     4
99999 - Dormant Company                                                      3
10710 - Manufacture of bread; manufacture of fresh pastry goods and cakes    3
71129 - Other engineering activities                                         2
59200 - Sound recording and music publishing activities                      2
None Supplied                                                                2
62020 - Information technology consultancy activities                        2
96090 - Other service activities n

### Observation

The random sample contains companies from a wide range of business activities, including construction, real estate, manufacturing, engineering, IT and professional services.

I also found two useful data-quality nuances. An `Active` Companies House status does not necessarily mean that a company is actively trading, as the sample includes a company classified as `99999 - Dormant Company`. In addition, `None Supplied` appears as a SIC value rather than a missing value, so a simple null check can overstate data completeness.

I keep these records in the sample and treat them as part of the real-world uncertainty in the source data.

### Check missing identification information

Company name, company number and address information may later help confirm whether a discovered VAT number belongs to the correct company. I therefore check how complete these fields are in the sample.

In [84]:
important_fields = [
    "CompanyName",
    "CompanyNumber",
    "RegAddress.AddressLine1",
    "RegAddress.PostTown",
    "RegAddress.PostCode",
    "SICCode.SicText_1"
]

missing_information = (
    company_sample[important_fields]
    .isna()
    .sum()
)

missing_information

CompanyName                0
CompanyNumber              0
RegAddress.AddressLine1    2
RegAddress.PostTown        3
RegAddress.PostCode        2
SICCode.SicText_1          0
dtype: int64

### Observation

The sample has complete company names and company numbers, while a small number of records have missing address information.

I keep these companies in the sample because incomplete address data is realistic and may affect how confidently a VAT number can be matched back to the correct company.

### Review company age


I look at the incorporation years to get a better sense of the sample and see whether it includes a mix of newer and older companies.

In [85]:
company_sample["IncorporationDate"] = pd.to_datetime(
    company_sample["IncorporationDate"],
    format="%d/%m/%Y",
    errors="coerce"
)

In [86]:
company_sample["IncorporationYear"] = (
    company_sample["IncorporationDate"].dt.year
)

company_sample["IncorporationYear"].describe()

count     100.000000
mean     2016.760000
std        11.429291
min      1962.000000
25%      2013.000000
50%      2020.000000
75%      2025.000000
max      2026.000000
Name: IncorporationYear, dtype: float64

### Observation

The sample includes both long-established and recently incorporated companies. Incorporation years range from 1962 to 2026, with a median of 2020.

This suggests that the random sample is not limited to either very new or very old businesses, although a large part of it consists of relatively recent companies.

## VAT discovery research

The next step is to investigate where UK VAT registration numbers can actually be discovered.

For each source, I want to understand:
- whether it contains VAT registration numbers;
- how reliably a VAT number can be linked to a specific company;
- what coverage it may provide;
- whether the source could realistically be used at scale;
- what can cause false positives or missing results.

### Initial source hypotheses

I identified several possible routes for VAT discovery:

1. **Official company websites**  
   VAT numbers may appear on legal pages, terms and conditions, invoices, ecommerce pages or other parts of a company's website.

2. **Government spending and procurement data**  
   Some public spending datasets contain both supplier names and VAT registration numbers.

3. **EORI numbers**  
   For UK VAT-registered businesses, the VAT number can be embedded in the EORI number, making EORI a possible source of VAT candidates.

4. **Bulk web data**  
   VAT numbers may exist across large numbers of webpages and documents, making web corpora potentially more suitable than crawling websites individually.

Each route will be tested separately rather than assumed to work.

### Prepare the VAT discovery results

I create a results table for the selected companies. For each company, I will record where I searched, any VAT candidate found, how it was verified, and the final decision.

In [87]:
vat_results = company_sample[
    [
        "CompanyName",
        "CompanyNumber",
        "RegAddress.AddressLine1",
        "RegAddress.PostTown",
        "RegAddress.PostCode"
    ]
].copy()

vat_results["Source"] = ""
vat_results["SourceURL"] = ""
vat_results["VATCandidate"] = ""
vat_results["HMRCValid"] = ""
vat_results["HMRCName"] = ""
vat_results["HMRCAddress"] = ""
vat_results["CompanyMatch"] = ""
vat_results["FinalStatus"] = "NOT CHECKED"
vat_results["Notes"] = ""

vat_results.head()

,CompanyName,CompanyNumber,RegAddress.AddressLine1,RegAddress.PostTown,RegAddress.PostCode,Source,SourceURL,VATCandidate,HMRCValid,HMRCName,HMRCAddress,CompanyMatch,FinalStatus,Notes
0,AJ PARTITIONS AND CEILINGS LTD,14816463,PARKINS ACCOUNTANTS,ROTHERHAM,S66 2BL,,,,,,,,NOT CHECKED,
1,ANDREWS RESIDENTIAL LIMITED,12574830,"OFFICE 5, MARLOWE HOUSE WATLING STREET",LEIGHTON BUZZARD,LU7 9LS,,,,,,,,NOT CHECKED,
2,AOB SUBSEA LTD,SC682059,4 WEST CRAIBSTONE STREET,ABERDEEN,AB11 6YL,,,,,,,,NOT CHECKED,
3,ARLINGTON GROUP ASSET MANAGEMENT LIMITED,02359077,15 WHITEHALL,LONDON,SW1A 2DD,,,,,,,,NOT CHECKED,
4,ASTRAZENECA UK LIMITED,03674842,1 FRANCIS CRICK AVENUE,CAMBRIDGE,CB2 0AA,,,,,,,,NOT CHECKED,


In [88]:
results_folder = Path("../results")

vat_results_file = results_folder / "vat_discovery_results.csv"

vat_results.to_csv(
    vat_results_file,
    index=False
)

print("Results file created:", vat_results_file)

Results file created: ..\results\vat_discovery_results.csv


### Test 1: Government spending data

I start by testing a public government spending dataset from DEFRA.

The dataset is useful for this experiment because it contains supplier names, supplier postcodes and VAT registration numbers. I want to measure how many usable VAT records it contains and whether any of the suppliers overlap with my random Companies House sample.

### Load the DEFRA dataset

The file could not be read using the default UTF-8 encoding. I therefore load it using Windows-1252 (`cp1252`), which correctly handles the characters in this dataset.

In [89]:
defra_file = original_data_folder / "defra_spending_january_2026.csv"

defra_data = pd.read_csv(
    defra_file,
    encoding="cp1252",
    skipinitialspace=True
)

print("Rows:", len(defra_data))
print("Columns:", len(defra_data.columns))

defra_data.head()

Rows: 921
Columns: 15


,Department,Entity,Date,Expense Type,Expense Area,Supplier,Transaction Number,Amount,PO Catergory Description,Supplier Postcode,Supplier Type,Contract Number,Project Code,Expenditure Type,Vat Registration Num
0,DE,DEFRA,08/01/2026,IA - POA & AUC - COST - ADDITIONS,FUTURE FARMING AND COUNTRYSIDE INITIATIVE,1SPATIAL GROUP LTD,1003292043,"£56,760.00",INTANGIBLE ASSETS (ICIP CAPEX),CB4 0WZ,SUPPLIER,C15417,DEFCOOD300216,Asset,GB100177077
1,DE,DEFRA,23/01/2026,EXP - PURCHASE OF GOODS/SERVICES - END USER SO...,RURAL PAYMENTS AGENCY INFO & TECH,1SPATIAL GROUP LTD,1003293216,"£880,800.00",IT SERVICES/SOFTWARE/HARDWARE,CB4 0WZ,SUPPLIER,32705,DEFCOOD301018,Expense,GB100177077
2,DE,DEFRA,30/01/2026,EXP - PURCHASE OF GOODS/SERVICES - RESEARCH & ...,"FOOD AND FARMING STRATEGY, INNOVATION, SYSTEMS",A H D B CEREALS,1003293873,"£30,000.00",CAPITAL R&D,CV8 2TL,NDPB,C30084,NaN,Expense,GB791452415
3,DE,Cefas,09/01/2026,Vessel Management,Corporate,A W Ship Management Ltd,20216612,"£355,181.11",Research Vessel Operations,EC2M 4TE,STANSME,3702,RV004,Research Vessel Operations,GB 283963068
4,DE,Cefas,15/01/2026,Vessel Management,Corporate,A W Ship Management Ltd,20216501,"£46,873.08",Specialist Project Materials,EC2M 4TE,STANSME,3702,RV004,Specialist Project Materials,GB 283963068


### Inspect the DEFRA data

I first review the available columns and the VAT field before trying to match suppliers to the Companies House sample.

In [90]:
defra_data.columns.tolist()

['Department',
 'Entity',
 'Date',
 'Expense Type',
 'Expense Area',
 'Supplier ',
 'Transaction Number',
 'Amount',
 'PO Catergory Description ',
 'Supplier Postcode',
 'Supplier Type',
 'Contract Number',
 'Project Code',
 'Expenditure Type',
 'Vat Registration Num']

In [91]:
defra_data.columns = defra_data.columns.str.strip()

defra_data.columns.tolist()

['Department',
 'Entity',
 'Date',
 'Expense Type',
 'Expense Area',
 'Supplier',
 'Transaction Number',
 'Amount',
 'PO Catergory Description',
 'Supplier Postcode',
 'Supplier Type',
 'Contract Number',
 'Project Code',
 'Expenditure Type',
 'Vat Registration Num']

### Check VAT number availability

I check how often a VAT registration number is provided in the DEFRA dataset before using it as a discovery source.

In [92]:
vat_column = "Vat Registration Num"

total_rows = len(defra_data)
rows_with_vat = defra_data[vat_column].notna().sum()

print("Total rows:", total_rows)
print("Rows with VAT:", rows_with_vat)
print(f"Share with VAT: {rows_with_vat / total_rows * 100:.1f}%")

Total rows: 921
Rows with VAT: 746
Share with VAT: 81.0%


In [93]:
defra_data["Vat Registration Num"].value_counts(
    dropna=False
).head(20)

Vat Registration Num
NaN               175
GB624298920        24
GB340316204        22
927 4872 86        17
GB424894330        16
GB524461265        15
1055 40018         14
GB 905 0549 42     13
20986 1253         12
GB 904443249       12
764244132          10
336 940192          9
GB 8727 99950       9
665 3009 41         9
118204348           8
888800181.          7
905280834           7
GB 523765636        7
GB 123382928        6
GB 618 1841 40      6
Name: count, dtype: int64

### Observation

Although 81% of the transaction rows contain a value in the VAT field, the numbers are not stored in a consistent format.

Some include the `GB` prefix, some contain spaces or punctuation, and some may not have the expected number of digits. Therefore, a populated VAT field does not necessarily mean that the value is immediately usable.

I normalize the values before measuring usable VAT coverage.

### Normalize the VAT numbers

I keep the original VAT value and create a separate normalized version.

For comparison purposes, I remove the `GB` prefix, spaces and punctuation, while keeping only the digits. I then check whether the resulting value contains 9 digits.

In [94]:
def normalize_vat_number(vat_value):
    if pd.isna(vat_value):
        return None

    vat_as_text = str(vat_value).upper().strip()

    if vat_as_text.startswith("GB"):
        vat_as_text = vat_as_text[2:]

    digits_only = "".join(
        character for character in vat_as_text
        if character.isdigit()
    )

    return digits_only if digits_only else None


defra_data["NormalizedVAT"] = (
    defra_data["Vat Registration Num"]
    .apply(normalize_vat_number)
)

In [95]:
defra_data[
    ["Vat Registration Num", "NormalizedVAT"]
].dropna().head(20)

,Vat Registration Num,NormalizedVAT
0,GB100177077,100177077
1,GB100177077,100177077
2,GB791452415,791452415
3,GB 283963068,283963068
4,GB 283963068,283963068
6,GB663603929,663603929
8,GB788629066,788629066
9,8888 00181,888800181
10,477763003,477763003
11,611853162,611853162


### Check the normalized VAT format

After normalization, I check how many VAT values contain exactly 9 digits.

This is only a format check. A 9-digit value is still only a VAT candidate until it is verified against HMRC.

In [96]:
defra_data["Has9DigitVAT"] = (
    defra_data["NormalizedVAT"]
    .str.fullmatch(r"\d{9}", na=False)
)

rows_with_9_digit_vat = defra_data["Has9DigitVAT"].sum()

print("Rows with a VAT value:", rows_with_vat)
print("Rows with a 9-digit VAT candidate:", rows_with_9_digit_vat)
print(
    f"Share of VAT values with 9 digits: "
    f"{rows_with_9_digit_vat / rows_with_vat * 100:.1f}%"
)

Rows with a VAT value: 746
Rows with a 9-digit VAT candidate: 729
Share of VAT values with 9 digits: 97.7%


### Observation

Most populated VAT values become valid 9-digit candidates after normalization. Of the 746 rows containing VAT information, 729 (97.7%) have exactly 9 digits after removing prefixes, spaces and punctuation.

This suggests that inconsistent formatting is more common than structurally unusable VAT data in this source. However, a correct 9-digit format does not prove that the VAT number is valid or belongs to the expected company.

### Review VAT values that do not match the expected format

I inspect the VAT values that do not result in exactly 9 digits after normalization to understand what kinds of data-quality problems remain.

In [97]:
unusual_vat_values = defra_data.loc[
    defra_data["Vat Registration Num"].notna()
    & ~defra_data["Has9DigitVAT"],
    [
        "Supplier",
        "Vat Registration Num",
        "NormalizedVAT"
    ]
]

unusual_vat_values

,Supplier,Vat Registration Num,NormalizedVAT
20,AMAZON WEB SERVICES EMEA SARL,LU 26888617,26888617
21,AMAZON WEB SERVICES EMEA SARL,LU 26888617,26888617
267,PACE-XL,GB7015920554,7015920554
328,SURREY HILLS NATIONAL LANDSCAPE,GB216 9472 4,21694724
336,THE FRESHWATER BIOLOGICAL ASSOCIATION,1530895238,1530895238
344,THE UNIVERSITY OF LEEDS,GB61351470,61351470
426,CELTIC DIAGNOSTICS LTD,IE95746910,95746910
427,CELTIC DIAGNOSTICS LTD,IE95746910,95746910
428,CELTIC DIAGNOSTICS LTD,IE95746910,95746910
435,DEPARTMENT FOR SCIENCE INNOVATION AND TECHNOLOGY,GB 888 8255,8888255


### Observation

The values that failed the 9-digit format check are not all the same type of error.

Some are foreign VAT numbers, such as values prefixed with `LU` or `IE`. Others appear incomplete or contain an unexpected number of digits. I also found a 12-digit GB value, which may represent a standard 9-digit UK VAT registration number followed by a branch or subsidiary identifier.

For the main UK analysis, I treat the standard 9-digit VAT registration number as the target format and keep non-standard values separate rather than automatically classifying them as invalid.

### Preserve the VAT country prefix

Because the dataset also contains foreign VAT numbers, I keep the country prefix separately instead of removing it completely during normalization. This helps distinguish UK VAT candidates from foreign VAT identifiers.

In [98]:
def get_vat_prefix(vat_value):
    if pd.isna(vat_value):
        return None

    vat_as_text = str(vat_value).upper().strip()

    letters = "".join(
        character for character in vat_as_text
        if character.isalpha()
    )

    return letters[:2] if letters else None


defra_data["VATPrefix"] = (
    defra_data["Vat Registration Num"]
    .apply(get_vat_prefix)
)

In [99]:
defra_data["VATPrefix"].value_counts(dropna=False)

VATPrefix
NaN    517
GB     399
IE       3
LU       2
Name: count, dtype: int64

In [100]:
unique_suppliers = defra_data["Supplier"].nunique()

suppliers_with_vat_candidate = (
    defra_data.loc[
        defra_data["Has9DigitVAT"],
        "Supplier"
    ]
    .nunique()
)

unique_vat_candidates = (
    defra_data.loc[
        defra_data["Has9DigitVAT"],
        "NormalizedVAT"
    ]
    .nunique()
)

print("Unique suppliers:", unique_suppliers)
print(
    "Suppliers with a 9-digit VAT candidate:",
    suppliers_with_vat_candidate
)
print(
    "Unique 9-digit VAT candidates:",
    unique_vat_candidates
)

print(
    f"Supplier-level VAT candidate coverage: "
    f"{suppliers_with_vat_candidate / unique_suppliers * 100:.1f}%"
)

Unique suppliers: 468
Suppliers with a 9-digit VAT candidate: 333
Unique 9-digit VAT candidates: 303
Supplier-level VAT candidate coverage: 71.2%


### Observation

The DEFRA dataset contains 468 unique suppliers. Of these, 333 have at least one 9-digit VAT candidate, giving a supplier-level VAT candidate coverage of 71.2%.

There are 308 unique VAT candidates for 333 suppliers with VAT information. This means that some VAT numbers are associated with more than one supplier name and should be investigated before treating the supplier-to-VAT relationship as one-to-one.

These figures measure candidate availability only. The VAT numbers have not yet been verified against HMRC.

### Check VAT numbers linked to multiple supplier names

Before using this source for company matching, I check whether the same VAT candidate is associated with multiple supplier names.

In [101]:
supplier_vat_pairs = (
    defra_data.loc[
        defra_data["Has9DigitVAT"],
        ["Supplier", "NormalizedVAT"]
    ]
    .drop_duplicates()
)

supplier_names_per_vat = (
    supplier_vat_pairs
    .groupby("NormalizedVAT")["Supplier"]
    .nunique()
    .sort_values(ascending=False)
)

supplier_names_per_vat.head(15)

NormalizedVAT
888800181    6
239503167    3
256435886    3
665300941    3
744492612    3
178178130    2
204046219    2
287461957    2
366927606    2
100132207    2
209861253    2
417632457    2
232327983    2
365969589    2
501624882    2
Name: Supplier, dtype: int64

In [102]:
vat_numbers_with_multiple_names = supplier_names_per_vat[
    supplier_names_per_vat > 1
].index

multiple_name_cases = (
    supplier_vat_pairs[
        supplier_vat_pairs["NormalizedVAT"].isin(
            vat_numbers_with_multiple_names
        )
    ]
    .sort_values("NormalizedVAT")
)

multiple_name_cases.head(30)

,Supplier,NormalizedVAT
240,NIAB EMR LTD,100132207
471,N I A B,100132207
254,NORTH PENNINES AONB PARTNERSHIP,178178130
101,DURHAM COUNTY COUNCIL,178178130
869,UPPER MEDWAY INTERNAL DRAINAGE BOARD,204046219
755,LOWER MEDWAY INTERNAL DRAINAGE BOARD,204046219
523,ATKINSREALIS UK LIMITED.,209861253
524,ATKINSRÉALIS UK LTD,209861253
26,ATOS IT SERVICES UK LTD,232327983
126,EVIDEN,232327983


### Observation

Several VAT candidates are associated with more than one supplier name.

Some cases appear to be simple naming variations, such as `RSK ADAS LTD`, `RSK ADAS LIMITED`, and `RSK ADAS Ltd`.

Other cases involve clearly different supplier names sharing the same VAT candidate. This is not necessarily a data error: UK companies can be registered as part of a VAT group, where multiple corporate entities use a single VAT registration number.

This means that VAT is a strong identifier for tax registration, but it does not always identify a single Companies House legal entity on its own. Additional company information may be needed when resolving VAT group members.

### Match the Companies House sample with DEFRA suppliers

I compare the 100 randomly selected companies with the suppliers found in the DEFRA dataset.

I start with simple exact matching after basic name cleaning. I prefer to miss a possible match rather than incorrectly assign a VAT number to the wrong company.

In [103]:
def clean_company_name(company_name):
    if pd.isna(company_name):
        return None

    company_name = str(company_name).upper()

    company_name = re.sub(
        r"[^A-Z0-9 ]",
        " ",
        company_name
    )

    company_name = re.sub(
        r"\s+",
        " ",
        company_name
    ).strip()

    return company_name

### Clean company names

I standardize company and supplier names before matching them. I remove punctuation and extra spaces and convert the text to uppercase.

In [104]:

def clean_company_name(company_name):
    if pd.isna(company_name):
        return None

    company_name = str(company_name).upper()

    company_name = re.sub(
        r"[^A-Z0-9 ]",
        " ",
        company_name
    )

    company_name = re.sub(
        r"\s+",
        " ",
        company_name
    ).strip()

    return company_name

### Apply the same cleaning rules

I apply the same name cleaning to both Companies House companies and DEFRA suppliers so they can be compared consistently.

In [105]:
company_sample["CleanCompanyName"] = (
    company_sample["CompanyName"]
    .apply(clean_company_name)
)

defra_data["CleanSupplierName"] = (
    defra_data["Supplier"]
    .apply(clean_company_name)
)

### Keep suppliers with VAT candidates

I keep only DEFRA suppliers that have a 9-digit VAT candidate, since these are the records that could provide a useful company-to-VAT match.

In [106]:
defra_suppliers_with_vat = (
    defra_data.loc[
        defra_data["Has9DigitVAT"],
        [
            "Supplier",
            "CleanSupplierName",
            "Supplier Postcode",
            "NormalizedVAT"
        ]
    ]
    .drop_duplicates()
)

### Match the sample with DEFRA suppliers

I compare the cleaned company names with the cleaned DEFRA supplier names using exact matching.

I use a conservative approach because assigning a VAT number to the wrong company would be worse than missing a possible match.

In [107]:
exact_matches = company_sample.merge(
    defra_suppliers_with_vat,
    left_on="CleanCompanyName",
    right_on="CleanSupplierName",
    how="inner"
)

print("Exact matches found:", len(exact_matches))

Exact matches found: 0


### Review the matches

I review any exact matches together with their postcodes and VAT candidates before deciding whether they are reliable.

In [108]:
exact_matches[
    [
        "CompanyName",
        "CompanyNumber",
        "Supplier",
        "NormalizedVAT",
        "RegAddress.PostCode",
        "Supplier Postcode"
    ]
]

,CompanyName,CompanyNumber,Supplier,NormalizedVAT,RegAddress.PostCode,Supplier Postcode


### Observation

No exact matches were found between the 100 Companies House companies and the DEFRA suppliers.

DEFRA contains useful VAT information for many of its own suppliers, but this individual dataset has very limited coverage for a random sample of UK companies. I therefore do not use it as the main VAT discovery source.

## Test 2: Official company websites

Company websites are a possible VAT discovery source because businesses may publish their VAT registration number on legal pages, terms and conditions, contact pages or other parts of their website.

I test this source on companies from the random sample to understand how often an official website can be identified and whether it exposes a VAT number.

### Select companies for the website test

I use the first 20 companies from the previously generated random sample. I do not select companies based on whether they have a known website or VAT number.

In [109]:
website_test_sample = company_sample.head(20).copy()

website_test_sample[
    ["CompanyName", "CompanyNumber", "RegAddress.PostCode"]
]

,CompanyName,CompanyNumber,RegAddress.PostCode
0,AJ PARTITIONS AND CEILINGS LTD,14816463,S66 2BL
1,ANDREWS RESIDENTIAL LIMITED,12574830,LU7 9LS
2,AOB SUBSEA LTD,SC682059,AB11 6YL
3,ARLINGTON GROUP ASSET MANAGEMENT LIMITED,02359077,SW1A 2DD
4,ASTRAZENECA UK LIMITED,03674842,CB2 0AA
5,AUDIO NUTRITION LTD,17027036,FY4 4HE
6,AV GLOBAL HOLDINGS LTD,17196916,PE2 9PX
7,AVTAR INVESTMENTS LTD,16376979,WV4 5EU
8,B & D ELIAS PROPERTIES LTD,12885766,SA8 4HU
9,BLAIRGOWRIE EVANGELICAL CHURCH SCIO,CS007183,NaN


### Prepare the website search results

For each company, I record whether I found an official website, whether a VAT candidate was present, where it was found and whether it was later verified.

In [110]:
website_results = website_test_sample[
    [
        "CompanyName",
        "CompanyNumber",
        "RegAddress.PostCode"
    ]
].copy()

website_results["Website"] = ""
website_results["VATCandidate"] = ""
website_results["VATSourcePage"] = ""
website_results["HMRCVerified"] = ""
website_results["FinalStatus"] = "NOT CHECKED"

website_results

,CompanyName,CompanyNumber,RegAddress.PostCode,Website,VATCandidate,VATSourcePage,HMRCVerified,FinalStatus
0,AJ PARTITIONS AND CEILINGS LTD,14816463,S66 2BL,,,,,NOT CHECKED
1,ANDREWS RESIDENTIAL LIMITED,12574830,LU7 9LS,,,,,NOT CHECKED
2,AOB SUBSEA LTD,SC682059,AB11 6YL,,,,,NOT CHECKED
3,ARLINGTON GROUP ASSET MANAGEMENT LIMITED,02359077,SW1A 2DD,,,,,NOT CHECKED
4,ASTRAZENECA UK LIMITED,03674842,CB2 0AA,,,,,NOT CHECKED
5,AUDIO NUTRITION LTD,17027036,FY4 4HE,,,,,NOT CHECKED
6,AV GLOBAL HOLDINGS LTD,17196916,PE2 9PX,,,,,NOT CHECKED
7,AVTAR INVESTMENTS LTD,16376979,WV4 5EU,,,,,NOT CHECKED
8,B & D ELIAS PROPERTIES LTD,12885766,SA8 4HU,,,,,NOT CHECKED
9,BLAIRGOWRIE EVANGELICAL CHURCH SCIO,CS007183,NaN,,,,,NOT CHECKED


Company: AJ PARTITIONS AND CEILINGS LTD
Official website: Not found
VAT candidate: Not found
Status: NOT FOUND
Notes: No clear official company website identified through initial web search.

Company: ANDREWS RESIDENTIAL LIMITED
Official website: Not found
VAT candidate: Not found
Status: NOT FOUND
Notes: A website with a similar name was found, but its legal information refers to a different registered company, so I rejected it as the official website.


Company: AOB SUBSEA LTD
Official website: Not found
VAT candidate: Not found
Status: NOT FOUND
Notes: No clear official company website identified through initial web search.

Company: ARLINGTON GROUP ASSET MANAGEMENT LIMITED
Official website: https://www.agam.co.uk/
VAT candidate: 69706407
Status: VAT CANDIDATE - FORMAT ISSUE
Notes: A VAT number was found in an official company document, but it contains only 8 digits and therefore requires further verification.

Company: ASTRAZENECA UK LIMITED
Official website: https://www.astrazeneca.co.uk/
VAT candidate: 582323642
Status: VAT CANDIDATE - NOT VERIFIED
Notes: A VAT number was found on an official AstraZeneca legal page and still needs to be verified against HMRC.

Company: AUDIO NUTRITION LTD
Official website: Not found
VAT candidate: Not found
Status: NOT FOUND
Notes: No clear official company website identified through initial web search.

Company: AV GLOBAL HOLDINGS LTD
Official website: Not found
VAT candidate: Not found
Status: NOT FOUND
Notes: No clear official company website identified through initial web search.

Company: AVATAR INVESTMENTS LTD
Official website: Not confirmed
VAT candidate: Not found
Status: UNRESOLVED
Notes: The company name produced several ambiguous search results, so I could not confidently identify the correct official website.

Company: B & D ELIAS PROPERTIES LTD
Official website: Not found
VAT candidate: Not found
Status: NOT FOUND
Notes: No clear official company website identified through initial web search.

Company: BLAIRGOWRIE EVANGELICAL CHURCH SCIO
Official website: https://www.hopeblair.com/
VAT candidate: Not found
Status: WEBSITE FOUND - VAT NOT FOUND
Notes: The official website was identified, but no VAT registration number was found on the pages checked.

Company: BLOOMSBURY COURT INTERIORS LIMITED
Official website: https://www.bloomsburycourtinteriors.co.uk/
VAT candidate: Not found
Status: WEBSITE FOUND - VAT NOT FOUND
Notes: The official website was confirmed using the company name and company number, but no VAT registration number was found on the pages checked.

Company: BRACI1 LIMITED
Official website: https://www.braci.co/
VAT candidate: Not found
Status: WEBSITE FOUND - VAT NOT FOUND
Notes: The website identifies itself as Braci1 Limited and provides the same company number, but no VAT registration number was found.

Company: BURLINGTON WELLESLEY SEARCH LIMITED
Official website: https://www.burlingtonwellesleysearch.co.uk/
VAT candidate: Not found
Status: WEBSITE FOUND - VAT NOT FOUND
Notes: An official company website was identified, but no VAT registration number was found through the initial website search.

Company: CAD CONSULTANTS LTD
Official website: Not found
VAT candidate: Not found
Status: NOT FOUND
Notes: No clear official company website identified through initial web search.

Company: CANDEY LIMITED
Official website: https://www.candey.com/
VAT candidate: Not found
Status: WEBSITE FOUND - VAT NOT FOUND
Notes: The official website was confirmed using the company name, company number and registered address, but no VAT registration number was found on the pages checked.

Company: CHARLIE PERKINS LIMITED
Official website: Not found
VAT candidate: Not found
Status: NOT FOUND
Notes: No clear official company website identified through initial web search. The company is also recorded as dormant in its latest accounts.

Company: CHRONOBAND104 LTD
Official website: Not found
VAT candidate: Not found
Status: NOT FOUND
Notes: No clear official company website identified through initial web search.

Company: CLARITY FINANCIAL LIMITED
Official website: https://clarityfin.co.uk/
VAT candidate: Not found
Status: WEBSITE FOUND - VAT NOT FOUND
Notes: The official website was identified, but the VAT number displayed on the site belongs to Sandringham Financial Partners Ltd, not Clarity Financial Limited, so I rejected it.

Company: CO BUILT FABRICATION LIMITED
Official website: https://www.co-built.net/
VAT candidate: Not found
Status: WEBSITE FOUND - VAT NOT FOUND
Notes: A website corresponding to the Co-Built fabrication business was identified, but no VAT registration number was found through the initial website search.

Company: CUBAN BOXING ACADEMY CIC
Official website: https://www.cubanboxingacademy.com/
VAT candidate: Not found
Status: WEBSITE FOUND - VAT NOT FOUND
Notes: The official website was identified using the company name and address, but no VAT registration number was found on the pages checked.

### Record the website search results

I organize the results of the manual website search in a structured table so I can compare the outcomes across the 20 companies.

In [111]:
website_results_data = [
    {
        "Company": "AJ PARTITIONS AND CEILINGS LTD",
        "OfficialWebsite": "",
        "VATCandidate": "",
        "Status": "NOT FOUND",
        "Notes": "No clear official company website identified through initial web search."
    },
    {
        "Company": "ANDREWS RESIDENTIAL LIMITED",
        "OfficialWebsite": "",
        "VATCandidate": "",
        "Status": "NOT FOUND",
        "Notes": "A similar website was found, but its legal information referred to a different registered company."
    },
    {
        "Company": "AOB SUBSEA LTD",
        "OfficialWebsite": "",
        "VATCandidate": "",
        "Status": "NOT FOUND",
        "Notes": "No clear official company website identified through initial web search."
    },
    {
        "Company": "ARLINGTON GROUP ASSET MANAGEMENT LIMITED",
        "OfficialWebsite": "https://www.agam.co.uk/",
        "VATCandidate": "69706407",
        "Status": "VAT CANDIDATE - FORMAT ISSUE",
        "Notes": "VAT number found in an official company document, but it contains only 8 digits."
    },
    {
        "Company": "ASTRAZENECA UK LIMITED",
        "OfficialWebsite": "https://www.astrazeneca.co.uk/",
        "VATCandidate": "582323642",
        "Status": "VAT CANDIDATE - NOT VERIFIED",
        "Notes": "VAT number found on an official company legal page."
    },
    {
        "Company": "AUDIO NUTRITION LTD",
        "OfficialWebsite": "",
        "VATCandidate": "",
        "Status": "NOT FOUND",
        "Notes": "No clear official company website identified through initial web search."
    },
    {
        "Company": "AV GLOBAL HOLDINGS LTD",
        "OfficialWebsite": "",
        "VATCandidate": "",
        "Status": "NOT FOUND",
        "Notes": "No clear official company website identified through initial web search."
    },
    {
        "Company": "AVATAR INVESTMENTS LTD",
        "OfficialWebsite": "",
        "VATCandidate": "",
        "Status": "UNRESOLVED",
        "Notes": "The company name produced ambiguous search results, so the official website could not be confirmed."
    },
    {
        "Company": "B & D ELIAS PROPERTIES LTD",
        "OfficialWebsite": "",
        "VATCandidate": "",
        "Status": "NOT FOUND",
        "Notes": "No clear official company website identified through initial web search."
    },
    {
        "Company": "BLAIRGOWRIE EVANGELICAL CHURCH SCIO",
        "OfficialWebsite": "https://www.hopeblair.com/",
        "VATCandidate": "",
        "Status": "WEBSITE FOUND - VAT NOT FOUND",
        "Notes": "Official website identified, but no VAT registration number was found."
    },
    {
        "Company": "BLOOMSBURY COURT INTERIORS LIMITED",
        "OfficialWebsite": "https://www.bloomsburycourtinteriors.co.uk/",
        "VATCandidate": "",
        "Status": "WEBSITE FOUND - VAT NOT FOUND",
        "Notes": "Official website identified, but no VAT registration number was found."
    },
    {
        "Company": "BRACI1 LIMITED",
        "OfficialWebsite": "https://www.braci.co/",
        "VATCandidate": "",
        "Status": "WEBSITE FOUND - VAT NOT FOUND",
        "Notes": "Official website identified, but no VAT registration number was found."
    },
    {
        "Company": "BURLINGTON WELLESLEY SEARCH LIMITED",
        "OfficialWebsite": "https://www.burlingtonwellesleysearch.co.uk/",
        "VATCandidate": "",
        "Status": "WEBSITE FOUND - VAT NOT FOUND",
        "Notes": "Official website identified, but no VAT registration number was found."
    },
    {
        "Company": "CAD CONSULTANTS LTD",
        "OfficialWebsite": "",
        "VATCandidate": "",
        "Status": "NOT FOUND",
        "Notes": "No clear official company website identified through initial web search."
    },
    {
        "Company": "CANDEY LIMITED",
        "OfficialWebsite": "https://www.candey.com/",
        "VATCandidate": "",
        "Status": "WEBSITE FOUND - VAT NOT FOUND",
        "Notes": "Official website identified, but no VAT registration number was found."
    },
    {
        "Company": "CHARLIE PERKINS LIMITED",
        "OfficialWebsite": "",
        "VATCandidate": "",
        "Status": "NOT FOUND",
        "Notes": "No clear official company website identified through initial web search."
    },
    {
        "Company": "CHRONOBAND104 LTD",
        "OfficialWebsite": "",
        "VATCandidate": "",
        "Status": "NOT FOUND",
        "Notes": "No clear official company website identified through initial web search."
    },
    {
        "Company": "CLARITY FINANCIAL LIMITED",
        "OfficialWebsite": "https://clarityfin.co.uk/",
        "VATCandidate": "",
        "Status": "WEBSITE FOUND - VAT NOT FOUND",
        "Notes": "A VAT number appeared on the website, but it belonged to another company, so it was rejected."
    },
    {
        "Company": "CO BUILT FABRICATION LIMITED",
        "OfficialWebsite": "https://www.co-built.net/",
        "VATCandidate": "",
        "Status": "WEBSITE FOUND - VAT NOT FOUND",
        "Notes": "Official website identified, but no VAT registration number was found."
    },
    {
        "Company": "CUBAN BOXING ACADEMY CIC",
        "OfficialWebsite": "https://www.cubanboxingacademy.com/",
        "VATCandidate": "",
        "Status": "WEBSITE FOUND - VAT NOT FOUND",
        "Notes": "Official website identified, but no VAT registration number was found."
    }
]

website_results = pd.DataFrame(website_results_data)

website_results

,Company,OfficialWebsite,VATCandidate,Status,Notes
0,AJ PARTITIONS AND CEILINGS LTD,,,NOT FOUND,No clear official company website identified t...
1,ANDREWS RESIDENTIAL LIMITED,,,NOT FOUND,"A similar website was found, but its legal inf..."
2,AOB SUBSEA LTD,,,NOT FOUND,No clear official company website identified t...
3,ARLINGTON GROUP ASSET MANAGEMENT LIMITED,https://www.agam.co.uk/,69706407,VAT CANDIDATE - FORMAT ISSUE,VAT number found in an official company docume...
4,ASTRAZENECA UK LIMITED,https://www.astrazeneca.co.uk/,582323642,VAT CANDIDATE - NOT VERIFIED,VAT number found on an official company legal ...
5,AUDIO NUTRITION LTD,,,NOT FOUND,No clear official company website identified t...
6,AV GLOBAL HOLDINGS LTD,,,NOT FOUND,No clear official company website identified t...
7,AVATAR INVESTMENTS LTD,,,UNRESOLVED,The company name produced ambiguous search res...
8,B & D ELIAS PROPERTIES LTD,,,NOT FOUND,No clear official company website identified t...
9,BLAIRGOWRIE EVANGELICAL CHURCH SCIO,https://www.hopeblair.com/,,WEBSITE FOUND - VAT NOT FOUND,"Official website identified, but no VAT regist..."


### Save the website search results

I save the manual website search results so they can be used later for the final analysis and VAT verification.

In [112]:
website_results_file = results_folder / "website_search_results.csv"

website_results.to_csv(
    website_results_file,
    index=False
)

print("Website results saved:", website_results_file)

Website results saved: ..\results\website_search_results.csv


### Observation

Official websites could be identified for only part of the sample, which already limits the coverage of this discovery method.

Even when a website was found, VAT information was not always published. I also found a case where a VAT number shown on the correct website belonged to another company mentioned on the page, showing the risk of accepting VAT numbers without further verification.

## Verify VAT candidates with HMRC

Any VAT number found during discovery is treated only as a candidate until it is checked against HMRC.

For each candidate, I verify:
- whether the VAT number is currently valid;
- the registered business name;
- the registered address;
- whether these details correspond to the Companies House company.

Company: ARLINGTON GROUP ASSET MANAGEMENT LIMITED
Official website: https://www.agam.co.uk/
VAT candidate: 69706407
HMRC verification: Failed - incorrect format
Status: NOT VERIFIED - FORMAT ISSUE
Notes: The number is presented as a VAT registration number in an official company document, but HMRC requires a 9-digit UK VAT number and does not accept this 8-digit value.

Company: ASTRAZENECA UK LIMITED
Official website: https://www.astrazeneca.co.uk/
VAT candidate: 582323642
HMRC verification: Valid
HMRC business name: ASTRAZENECA UK LIMITED
HMRC address: 1 FRANCIS CRICK AVENUE, CB2 0AA, GB
Company match: Yes
Status: VERIFIED
Notes: The VAT number was found on the official company website and independently confirmed through HMRC. The registered business name returned by HMRC matches the target company.

Check the company address 

In [113]:
company_sample[
    company_sample["CompanyName"] == "ASTRAZENECA UK LIMITED"
][
    [
        "CompanyName",
        "CompanyNumber",
        "RegAddress.AddressLine1",
        "RegAddress.AddressLine2",
        "RegAddress.PostTown",
        "RegAddress.PostCode"
    ]
]

,CompanyName,CompanyNumber,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.PostCode
4,ASTRAZENECA UK LIMITED,03674842,1 FRANCIS CRICK AVENUE,CAMBRIDGE BIOMEDICAL CAMPUS,CAMBRIDGE,CB2 0AA


### Observation

The AstraZeneca VAT candidate was successfully verified. HMRC confirmed both the company name and registered address, matching the Companies House record.